# Step 5 - Sentiment Preservation (XLM-RoBERTa)

Notebook nay chi lam 2 phan ban yeu cau:

1. Chay sentiment inference bang XLM-RoBERTa cho 4 cot: `source_vi`, `reference_en`, `gpt_en`, `gemini_en`.
2. Tinh overall sentiment preservation accuracy theo `source_vi` cho GPT va Gemini.

Output files:
- `outputs_medev/sentiment_predictions_xlmr.csv`
- `outputs_medev/sentiment_preservation_overall.csv`

In [1]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd


def ensure_module(module_name: str, package_name: str = None):
    package_name = package_name or module_name
    try:
        __import__(module_name)
    except ModuleNotFoundError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


ensure_module("transformers")
ensure_module("torch")

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

OUT_DIR = "outputs_medev"
GPT_PATH = os.path.join(OUT_DIR, "translations_gpt.csv")
GEMINI_PATH = os.path.join(OUT_DIR, "translations_gemini.csv")

PRED_PATH = os.path.join(OUT_DIR, "sentiment_predictions_xlmr.csv")
SUMMARY_PATH = os.path.join(OUT_DIR, "sentiment_preservation_overall.csv")

os.makedirs(OUT_DIR, exist_ok=True)
print("Setup complete")

c:\Anaconda\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete


In [2]:
gpt_df = pd.read_csv(GPT_PATH)
gemini_df = pd.read_csv(GEMINI_PATH)

# Keep successful translations only
mask_gpt_ok = ~gpt_df["translated_en"].astype(str).str.startswith("ERROR", na=False)
mask_gem_ok = ~gemini_df["translated_en"].astype(str).str.startswith("ERROR", na=False)

gpt_ok = gpt_df.loc[mask_gpt_ok, ["id", "source_vi", "reference_en", "translated_en"]].copy()
gem_ok = gemini_df.loc[mask_gem_ok, ["id", "translated_en"]].copy()

paired = gpt_ok.rename(columns={"translated_en": "gpt_en"}).merge(
    gem_ok.rename(columns={"translated_en": "gemini_en"}),
    on="id",
    how="inner",
)
paired = paired.sort_values("id").reset_index(drop=True)

print(f"GPT successful rows: {len(gpt_ok):,}")
print(f"Gemini successful rows: {len(gem_ok):,}")
print(f"Paired rows for sentiment comparison: {len(paired):,}")
paired.head(3)

GPT successful rows: 8,959
Gemini successful rows: 8,959
Paired rows for sentiment comparison: 8,959


,id,source_vi,reference_en,gpt_en,gemini_en
0,1,Thực trạng kiến thức và thực hành của người có...,"Knowledge, practices in public health service ...",The current status of knowledge and practices ...,Current status of knowledge and practices of h...
1,2,"Mô tả thực trạng kiến thức, thực hành của ngườ...","Describe knowledge, practices in public health...",Describe the current status of knowledge and p...,Describe the current status of knowledge and p...
2,3,Phương pháp: Thiết kế nghiên mô tả cắt ngang đ...,Methodology: A cross sectional study was used ...,Methods: A cross-sectional descriptive study w...,Methods: A cross-sectional descriptive study w...


In [3]:
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
DEVICE = 0 if torch.cuda.is_available() else -1

print(f"Loading model: {MODEL_NAME}")
print(f"Device: {'cuda' if DEVICE == 0 else 'cpu'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

sentiment_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    truncation=True,
    max_length=256,
)


def normalize_label(raw_label: str) -> str:
    lbl = str(raw_label).strip().lower()
    mapping = {
        "negative": "negative",
        "neutral": "neutral",
        "positive": "positive",
        "neg": "negative",
        "neu": "neutral",
        "pos": "positive",
        "label_0": "negative",
        "label_1": "neutral",
        "label_2": "positive",
        "0": "negative",
        "1": "neutral",
        "2": "positive",
    }

    if lbl in mapping:
        return mapping[lbl]

    if "negative" in lbl or "label_0" in lbl:
        return "negative"
    if "neutral" in lbl or "label_1" in lbl:
        return "neutral"
    if "positive" in lbl or "label_2" in lbl:
        return "positive"

    return "unknown"


print("XLM-RoBERTa sentiment pipeline ready")

Loading model: cardiffnlp/twitter-xlm-roberta-base-sentiment
Device: cuda


Device set to use cuda:0


XLM-RoBERTa sentiment pipeline ready


In [4]:
BATCH_SIZE = 32


def infer_labels(text_series: pd.Series, batch_size: int = 32):
    texts = text_series.fillna("").astype(str).tolist()
    raw = sentiment_pipe(texts, batch_size=batch_size, truncation=True, max_length=256)
    labels = [normalize_label(x.get("label", "")) for x in raw]
    scores = [float(x.get("score", np.nan)) for x in raw]
    return labels, scores


for col, prefix in [
    ("source_vi", "source"),
    ("reference_en", "reference"),
    ("gpt_en", "gpt"),
    ("gemini_en", "gemini"),
]:
    print(f"Inferring sentiment for {col} ...")
    labels, scores = infer_labels(paired[col], batch_size=BATCH_SIZE)
    paired[f"{prefix}_sentiment"] = labels
    paired[f"{prefix}_sentiment_score"] = scores

unknown_stats = {
    "source_unknown": int((paired["source_sentiment"] == "unknown").sum()),
    "reference_unknown": int((paired["reference_sentiment"] == "unknown").sum()),
    "gpt_unknown": int((paired["gpt_sentiment"] == "unknown").sum()),
    "gemini_unknown": int((paired["gemini_sentiment"] == "unknown").sum()),
}

paired.to_csv(PRED_PATH, index=False, encoding="utf-8")
print(f"Saved sentiment predictions -> {PRED_PATH}")
print("Unknown-label stats:", unknown_stats)
paired[["id", "source_sentiment", "gpt_sentiment", "gemini_sentiment"]].head(5)

Inferring sentiment for source_vi ...
Inferring sentiment for reference_en ...
Inferring sentiment for gpt_en ...
Inferring sentiment for gemini_en ...
Saved sentiment predictions -> outputs_medev\sentiment_predictions_xlmr.csv
Unknown-label stats: {'source_unknown': 0, 'reference_unknown': 0, 'gpt_unknown': 0, 'gemini_unknown': 0}


,id,source_sentiment,gpt_sentiment,gemini_sentiment
0,1,neutral,neutral,neutral
1,2,neutral,neutral,neutral
2,3,neutral,neutral,neutral
3,4,neutral,neutral,neutral
4,5,neutral,neutral,neutral


In [5]:
# Step 2: Overall sentiment preservation accuracy vs source_vi
base = paired[paired["source_sentiment"] != "unknown"].copy()

gpt_eval = base[base["gpt_sentiment"] != "unknown"].copy()
gem_eval = base[base["gemini_sentiment"] != "unknown"].copy()


def preservation_summary(eval_df: pd.DataFrame, model_col: str, model_name: str) -> dict:
    if len(eval_df) == 0:
        return {
            "Model": model_name,
            "ComparedRows": 0,
            "MatchedRows": 0,
            "PreservationAccuracy": np.nan,
            "PreservationAccuracyPct": np.nan,
        }

    matched = (eval_df[model_col] == eval_df["source_sentiment"]).sum()
    acc = matched / len(eval_df)
    return {
        "Model": model_name,
        "ComparedRows": int(len(eval_df)),
        "MatchedRows": int(matched),
        "PreservationAccuracy": round(float(acc), 6),
        "PreservationAccuracyPct": round(float(acc) * 100, 3),
    }


summary = pd.DataFrame([
    preservation_summary(gpt_eval, "gpt_sentiment", "GPT-5.2"),
    preservation_summary(gem_eval, "gemini_sentiment", "Gemini-2.5-Flash"),
])

summary.to_csv(SUMMARY_PATH, index=False, encoding="utf-8")
print(f"Saved overall preservation summary -> {SUMMARY_PATH}")
summary

Saved overall preservation summary -> outputs_medev\sentiment_preservation_overall.csv


,Model,ComparedRows,MatchedRows,PreservationAccuracy,PreservationAccuracyPct
0,GPT-5.2,8959,7649,0.853778,85.378
1,Gemini-2.5-Flash,8959,7674,0.856569,85.657


## RQ2 - Preservation Accuracy by Sentiment Class

Bang nay tinh do bao toan sentiment theo tung lop `negative / neutral / positive`, lay `source_sentiment` lam nhan goc.

In [6]:
BY_CLASS_PATH = os.path.join(OUT_DIR, "sentiment_preservation_by_class.csv")

classes = ["negative", "neutral", "positive"]
rows = []

base_cls = paired[paired["source_sentiment"] != "unknown"].copy()

for cls in classes:
    sub = base_cls[base_cls["source_sentiment"] == cls].copy()
    n = len(sub)

    gpt_match = int((sub["gpt_sentiment"] == sub["source_sentiment"]).sum())
    gemini_match = int((sub["gemini_sentiment"] == sub["source_sentiment"]).sum())

    gpt_acc = (gpt_match / n) if n > 0 else np.nan
    gemini_acc = (gemini_match / n) if n > 0 else np.nan

    rows.append(
        {
            "Class": cls,
            "SourceCount": int(n),
            "GPT_Matched": gpt_match,
            "GPT_Accuracy": round(float(gpt_acc), 6) if n > 0 else np.nan,
            "GPT_AccuracyPct": round(float(gpt_acc) * 100, 3) if n > 0 else np.nan,
            "Gemini_Matched": gemini_match,
            "Gemini_Accuracy": round(float(gemini_acc), 6) if n > 0 else np.nan,
            "Gemini_AccuracyPct": round(float(gemini_acc) * 100, 3) if n > 0 else np.nan,
            "Delta_Gemini_minus_GPT_pp": round((float(gemini_acc) - float(gpt_acc)) * 100, 3) if n > 0 else np.nan,
        }
    )

by_class = pd.DataFrame(rows)

# Macro-average (unweighted across classes)
macro_gpt = by_class["GPT_Accuracy"].mean()
macro_gem = by_class["Gemini_Accuracy"].mean()

# Overall (weighted by class size)
overall_n = len(base_cls)
overall_gpt_match = int((base_cls["gpt_sentiment"] == base_cls["source_sentiment"]).sum())
overall_gem_match = int((base_cls["gemini_sentiment"] == base_cls["source_sentiment"]).sum())
overall_gpt = overall_gpt_match / overall_n if overall_n > 0 else np.nan
overall_gem = overall_gem_match / overall_n if overall_n > 0 else np.nan

extra_rows = pd.DataFrame(
    [
        {
            "Class": "macro_avg",
            "SourceCount": int(overall_n),
            "GPT_Matched": np.nan,
            "GPT_Accuracy": round(float(macro_gpt), 6),
            "GPT_AccuracyPct": round(float(macro_gpt) * 100, 3),
            "Gemini_Matched": np.nan,
            "Gemini_Accuracy": round(float(macro_gem), 6),
            "Gemini_AccuracyPct": round(float(macro_gem) * 100, 3),
            "Delta_Gemini_minus_GPT_pp": round((float(macro_gem) - float(macro_gpt)) * 100, 3),
        },
        {
            "Class": "overall_weighted",
            "SourceCount": int(overall_n),
            "GPT_Matched": overall_gpt_match,
            "GPT_Accuracy": round(float(overall_gpt), 6),
            "GPT_AccuracyPct": round(float(overall_gpt) * 100, 3),
            "Gemini_Matched": overall_gem_match,
            "Gemini_Accuracy": round(float(overall_gem), 6),
            "Gemini_AccuracyPct": round(float(overall_gem) * 100, 3),
            "Delta_Gemini_minus_GPT_pp": round((float(overall_gem) - float(overall_gpt)) * 100, 3),
        },
    ]
)

rq2_table = pd.concat([by_class, extra_rows], ignore_index=True)
rq2_table.to_csv(BY_CLASS_PATH, index=False, encoding="utf-8")

print(f"Saved RQ2 table -> {BY_CLASS_PATH}")
rq2_table

Saved RQ2 table -> outputs_medev\sentiment_preservation_by_class.csv


,Class,SourceCount,GPT_Matched,GPT_Accuracy,GPT_AccuracyPct,Gemini_Matched,Gemini_Accuracy,Gemini_AccuracyPct,Delta_Gemini_minus_GPT_pp
0,negative,1352,1104.0,0.816568,81.657,1117.0,0.826183,82.618,0.962
1,neutral,7408,6419.0,0.866496,86.650,6429.0,0.867846,86.785,0.135
2,positive,199,126.0,0.633166,63.317,128.0,0.643216,64.322,1.005
3,macro_avg,8959,NaN,0.772077,77.208,NaN,0.779082,77.908,0.701
4,overall_weighted,8959,7649.0,0.853778,85.378,7674.0,0.856569,85.657,0.279
